In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Config, GPT2LMHeadModel, GPT2Tokenizer
from transformers import AdamW, get_linear_schedule_with_warmup

# Custom dataset
class NumberSequenceDataset(Dataset):
    def __init__(self, tokenizer):
        self.data = " ".join(map(str, range(10)))
        self.tokenizer = tokenizer

    def __len__(self):
        return 1

    def __getitem__(self, idx):
        return self.tokenizer.encode(self.data, return_tensors="pt")[0]

# Create custom tokenizer
vocab = {str(i): i for i in range(10)}
vocab['<|endoftext|>'] = 10  # Special token
tokenizer = GPT2Tokenizer(vocab_file=None, tokenizer_file=None)
tokenizer.vocab = vocab
tokenizer.ids_to_tokens = {v: k for k, v in vocab.items()}

# Configure a tiny GPT-2 model
config = GPT2Config(
    vocab_size=11,  # 0-9 + special token
    n_positions=20,
    n_ctx=20,
    n_embd=32,
    n_layer=2,
    n_head=2,
    bos_token_id=tokenizer.vocab['<|endoftext|>'],
    eos_token_id=tokenizer.vocab['<|endoftext|>'],
)

# Create model
model = GPT2LMHeadModel(config)

# Prepare dataset and dataloader
dataset = NumberSequenceDataset(tokenizer)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# Training settings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=1000)

# Training loop
model.train()
for epoch in range(100):
    for batch in dataloader:
        inputs = batch.to(device)
        outputs = model(inputs, labels=inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

# Test the model
model.eval()
input_text = "0 1 2 3"
input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)
with torch.no_grad():
    output = model.generate(input_ids, max_length=20, num_return_sequences=1, no_repeat_ngram_size=0)
generated_text = tokenizer.decode(output[0])
print(f"Input: {input_text}")
print(f"Generated: {generated_text}")

ModuleNotFoundError: No module named 'torch'

In [ ]:
func TestSimple(t *testing.T) {
    // Define vocabulary
    vocab := []string{"0", "1", "2", "3", "4", "5", "6", "7", "8", "9"}
    
    // Create new GPT2 model
    gpt2 := newGPT2(10, len(vocab), 4, 4, 32, vocab)
    
    // Initialize weights
    initializeWeights(gpt2)
    
    // Test inference
    outp, err := gpt2.Inference("12345678", 1, 9)
    if err != nil {
        t.Fatalf("Inference failed: %v", err)
    }
    
    // Encode training data
    data, err := gpt2.Tokenizer.Encode("0123456789101112131415161718192021222324252627282930313233343536373839404142434445464748495051525354555657585960616263646566676869707172737475767778798081828384858687888990919293949596979899100101102103104105106107108109110111112113114115116117118119120121122123124125126127128129130131132133134135136137138139140141142143144145146147148149150151152153154155156157158159160161162163164165166167168169170171172173174175176177178179180181182183184185186187188189190191192193194195196197198199200201202203204205206207208209210211212213214215216217218219220221222223224225226227228229230231232233234235236237238239240241242243244245246247248249250251252253254255256257258259260261262263264265266267268269270271272273274275276277278279280281282283284285286287288289290291292293294295296297298299")
    if err != nil {
        t.Fatalf("Encoding failed: %v", err)
    }
    
    // Create data loaders
    loader, err := newDataLoaderFromInts(data, 1, 9)
    if err != nil {
        t.Fatalf("Failed to create loader: %v", err)
    }
    loader2, err := newDataLoaderFromInts(data, 1, 9)
    if err != nil {
        t.Fatalf("Failed to create loader2: %v", err)
    }
    
    // Train the model
    if err := gpt2.Train(loader, loader2, 1, 9); err != nil {
        t.Fatalf("Training failed: %v", err)
    }
    
    fmt.Println("Initial output:", outp)
    
    // Test inference after training
    outpAfterTraining, err := gpt2.Inference("12345678", 1, 9)
    if err != nil {
        t.Fatalf("Inference after training failed: %v", err)
    }
    
    fmt.Println("Output after training:", outpAfterTraining)
}

func initializeWeights(model *GPT2) {
    // Initialize weights using Xavier/Glorot initialization
    for i := range model.Params.Memory {
        model.Params.Memory[i] = float32(rand.NormFloat64()) * sqrt(2.0 / float32(model.Config.C))
    }
}

func sqrt(x float32) float32 {
    return float32(math.Sqrt(float64(x)))
}